In [1]:
import json
from pathlib import Path

import faiss
import numpy as np
import torch
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

DOCS_DIR = "documents"
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
GEN_MODEL_NAME = "google/flan-t5-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


C:\Users\Aryan\Desktop\Celebal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


In [2]:
SUPPORTED_EXTENSIONS = {".txt", ".md", ".pdf"}


def read_pdf(path):
    reader = PdfReader(str(path))
    return "\n\n".join(page.extract_text() or "" for page in reader.pages)


def load_documents(directory):
    directory = Path(directory)
    documents = []
    for path in sorted(directory.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue
        text = read_pdf(path) if path.suffix.lower() == ".pdf" else path.read_text(encoding="utf-8", errors="ignore")
        if text.strip():
            documents.append({"source": str(path.relative_to(directory)), "text": text})
    return documents


documents = load_documents(DOCS_DIR)
print(f"Loaded {len(documents)} document(s): {[d['source'] for d in documents]}")

Loaded 1 document(s): ['sample_handbook.txt']


In [3]:
def chunk_text(text, chunk_size=800, overlap=150):
    text = " ".join(text.split())
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + chunk_size, n)
        if end < n:
            boundary = text.rfind(". ", start + int(chunk_size * 0.5), end)
            if boundary != -1:
                end = boundary + 1
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= n:
            break
        start = max(end - overlap, start + 1)
    return chunks


all_chunks, metadata = [], []
for doc in documents:
    for i, chunk in enumerate(chunk_text(doc["text"])):
        all_chunks.append(chunk)
        metadata.append({"source": doc["source"], "chunk_id": i, "text": chunk})

print(f"Split into {len(all_chunks)} chunks")

Split into 5 chunks


In [4]:
embedder = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)


def embed(texts):
    return embedder.encode(
        texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")


chunk_embeddings = embed(all_chunks)
print(f"Embedded {chunk_embeddings.shape[0]} chunks into {chunk_embeddings.shape[1]}-dim vectors")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5704.12it/s]

Embedded 5 chunks into 384-dim vectors


In [5]:
index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings)
print(f"FAISS index built with {index.ntotal} vectors")


def search(query, top_k=3):
    query_embedding = embed([query])
    scores, indices = index.search(query_embedding, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        entry = dict(metadata[idx])
        entry["score"] = float(score)
        results.append(entry)
    return results

FAISS index built with 5 vectors


In [6]:
PROMPT_TEMPLATE = """Answer the question using only the context below. \
If the answer is not contained in the context, say "I don't know based on the provided documents."

Context:
{context}

Question: {question}
Answer:"""

tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL_NAME).to(DEVICE)


def generate(question, context_chunks, max_context_chars=2000):
    context = "\n\n".join(context_chunks)[:max_context_chars]
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        output_ids = gen_model.generate(**inputs, max_new_tokens=200)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 4999.80it/s]


[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [7]:
def answer(question, top_k=3, verbose=True):
    results = search(question, top_k=top_k)
    context_chunks = [r["text"] for r in results]
    answer_text = generate(question, context_chunks)

    if verbose:
        print(f"Q: {question}")
        print(f"A: {answer_text}\n")
        print("Sources:")
        for r in results:
            preview = r["text"][:120].replace("\n", " ")
            print(f"  - {r['source']} (chunk {r['chunk_id']}, score={r['score']:.3f}): {preview}...")
        print()

    return {"answer": answer_text, "sources": results}

In [8]:
_ = answer("How many privilege leaves do employees get per year?")
_ = answer("What is the IT helpdesk extension number?")
_ = answer("What is the capital of France?")

Q: How many privilege leaves do employees get per year?
A: 18 days

Sources:
  - sample_handbook.txt (chunk 1, score=0.492): are permitted with manager approval. Leave Policy Employees accrue 18 days of Privilege Leave (PL) per calendar year, cr...
  - sample_handbook.txt (chunk 0, score=0.386): NovaTech Solutions — Employee Handbook (Excerpt) Company Overview NovaTech Solutions is a mid-sized software consultancy...
  - sample_handbook.txt (chunk 2, score=0.322): ory in-office days). Fully remote work may be approved by a department head for up to 4 weeks per year, for reasons such...



Q: What is the IT helpdesk extension number?
A: 4040

Sources:
  - sample_handbook.txt (chunk 4, score=0.398): ary action, up to and including termination. Conflicts of interest, including outside employment with competitors, must ...
  - sample_handbook.txt (chunk 0, score=0.279): NovaTech Solutions — Employee Handbook (Excerpt) Company Overview NovaTech Solutions is a mid-sized software consultancy...
  - sample_handbook.txt (chunk 3, score=0.204): ncryption and an approved antivirus solution enabled at all times. Employees must not store client data on personal devi...



Q: What is the capital of France?
A: I don't know

Sources:
  - sample_handbook.txt (chunk 0, score=-0.014): NovaTech Solutions — Employee Handbook (Excerpt) Company Overview NovaTech Solutions is a mid-sized software consultancy...
  - sample_handbook.txt (chunk 1, score=-0.058): are permitted with manager approval. Leave Policy Employees accrue 18 days of Privilege Leave (PL) per calendar year, cr...
  - sample_handbook.txt (chunk 4, score=-0.066): ary action, up to and including termination. Conflicts of interest, including outside employment with competitors, must ...



In [9]:
question = "What is the monthly internet bill reimbursement limit?"
_ = answer(question)

Q: What is the monthly internet bill reimbursement limit?
A: INR 1,500

Sources:
  - sample_handbook.txt (chunk 2, score=0.612): ory in-office days). Fully remote work may be approved by a department head for up to 4 weeks per year, for reasons such...
  - sample_handbook.txt (chunk 0, score=0.222): NovaTech Solutions — Employee Handbook (Excerpt) Company Overview NovaTech Solutions is a mid-sized software consultancy...
  - sample_handbook.txt (chunk 3, score=0.220): ncryption and an approved antivirus solution enabled at all times. Employees must not store client data on personal devi...

